In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [4]:
processed_dir = Path(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\data\processed')

tfidf_matrix_bigrams = load_npz(processed_dir / "tfidf_matrix_bigrams.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
df = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert tfidf_matrix_bigrams.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(df["id"])

print("Shape :", tfidf_matrix_bigrams.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LN\\Documents\\Jedha\\Projets\\NORTH_FACE\\Helene\\data\\processed\\sample_clean.csv'

In [ ]:
processed_dir = Path("../data/processed")

tfidf_matrix = load_npz(processed_dir / "tfidf_matrix.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
df = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert tfidf_matrix.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(df["id"])

print("Dimensions :", tfidf_matrix.shape)

In [ ]:
pd.DataFrame(
    tfidf_matrix[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [ ]:
db = dbscan = DBSCAN(
    eps=0.5,
    min_samples=10,
    metric="cosine"
)

In [ ]:
db.fit_predict(tfidf_matrix)

data["cluster"] = db.labels_

display(data["cluster"].value_counts().sort_index())

In [ ]:
## Import go to build our own figure
import plotly.graph_objects as go

In [ ]:
display(data.sort_values("cluster"))

In [ ]:
print(f"clusters number : {len(set(data['cluster'])) - 1}")  # On soustrait 1 pour exclure le bruit (-1)

Le cluster -1 est le cluster considéré comme du bruit par l'algorithme Dbscan. Nous allons tenter de réduire le nombre de produits dans ce cluster en ajustant les paramètres de l'algorithme.

In [ ]:
parameters = {
    "eps": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "min_samples": [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
}

results = []

for params in ParameterGrid(parameters):
    clusters = DBSCAN(**params, metric="cosine").fit_predict(tfidf_matrix)
    without_noise = clusters != -1
    try : 
        score = silhouette_score(
            tfidf_matrix[without_noise],
            clusters[without_noise],
            metric="cosine"
        )
    except ValueError:
        score = np.nan

    results.append({
        **params,
        "silhouette": score,
        "noise_pct": (1 - without_noise.mean()) * 100,
        "clusters_nb": len(set(clusters)) - (1 if -1 in clusters else 0)
    })

results = pd.DataFrame(results)
# display(results.sort_values("clusters_nb", ascending=False))

In [ ]:
# candidates = results[(results["clusters_nb"] <= 20) & (results["clusters_nb"] >= 10) & results["silhouette"].notna()]

In [ ]:
candidates = results[(results["clusters_nb"].between(10, 20)) & (results["noise_pct"] < 30)].sort_values("silhouette", ascending=False)
candidates

In [ ]:

best = candidates.sort_values("silhouette", ascending=False).iloc[0]
display(best)

In [ ]:
# import matplotlib.pyplot as plt

# plt.scatter(results["%_bruit"], results["silhouette"])
# plt.xlabel("Proportion de bruit")
# plt.ylabel("Score de silhouette")
# plt.show()

In [ ]:
# import numpy as np
# import plotly.graph_objects as go
fig = go.Figure()

# Loop through each label for our cluster
for i in np.unique(db.labels_):
    label = tfidf_matrix[db.labels_ == i]
    fig.add_trace(go.Scatter(x=results["noise_pct"], y=results["silhouette"], mode="markers", name="Cluster {}".format(i)))
    fig.update_layout(
    xaxis_title="Noise percentage)",
    yaxis_title="Score de silhouette")
fig.show()

Le code trace les mêmes points (results["noise_pct"], results["silhouette"]) à chaque itération de la boucle : label n’est jamais utilisé, donc tous les “clusters” sont identiques et superposés. Ces données de results décrivent les essais de paramètres DBSCAN, pas les clusters.

Pour visualiser les clusters, il faut réduire tfidf_matrix en 2D, par exemple :

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import TruncatedSVD

coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(tfidf_matrix)

fig = go.Figure()
for i in np.unique(db.labels_):
    mask = db.labels_ == i
    fig.add_trace(go.Scatter(x=coords[mask, 0], y=coords[mask, 1], mode="markers", name=f"Cluster {i}"))
fig.show()

# 